## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [1]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

C:\Users\Vish\AppData\Local\Temp\ipykernel_11636\3762973587.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


### 1 & 2. Model and Chat Model 

In [33]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [34]:
def get_chat_llm (params=None):
    
    llm = ChatGoogleGenerativeAI(
        model = "gemini-2.5-flash",
        temperature=0.3,
        api_key=Model_API
    )
    
    return llm

In [35]:
llm_model = get_chat_llm()

### 3. Chat Message 

In [5]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [6]:
print(message)

content='For a quintessential sport sedan, consider the **BMW M3**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e8bcc-7968-7ea3-887e-fb66a60c74da-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 33, 'output_tokens': 169, 'total_tokens': 202, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 156}}


### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [7]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [8]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.3, client=<google.genai.client.Client object at 0x000001F46FED8150>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

#### Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [10]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [11]:
response = llm_model.invoke(formatted_prompt)
response

AIMessage(content="The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).\n\n---\n\nThe **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider':

In [12]:
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).

---

The **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.


In [13]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**.

---

The **Toyota Supra** is an iconic Japanese sports car known for its performance, rear-wheel drive layout, and powerful inline-six engines. The **fourth-generation (Mk4)**, produced from 1993-2002, is particularly legendary, largely due to its highly tunable 2JZ-GTE engine and appearances in popular culture like the *Fast & Furious* film series. After a long hiatus, the Supra returned in 2019 (the **fifth-generation or A90/A91**) as a collaboration with BMW, maintaining its focus on performance and driver engagement.


### 5. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [14]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [15]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [16]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:


### 6. Output Parsers

Output parsers are used to convert the raw responses generated by an LLM into a more structured and useful format. They are especially helpful when working with structured data or when standardizing outputs from different language and chat models.

LangChain provides a variety of output parsers to handle different formatting needs. In this lab, you will work with the following two parsers:

* JSON Parser: Produces output in JSON format according to a specified schema. It can also utilize a Pydantic model to generate structured JSON data, making it one of the most dependable options for obtaining structured output without relying on function calling.

* CSV Parser: Formats the model's output as a list of comma-separated values (CSV), which is useful for tabular data representation.

In [17]:
class Car(BaseModel):
    query:str = Field(description="question to setup a car")
    response:str = Field(description="answer to resolve the car")

In [18]:
car_query = "tell me the information about Lambogini"

output_parser = JsonOutputParser(pydantic_object=Car)
format_instruction = output_parser.get_format_instructions()

prompt  = PromptTemplate(
    template= "Answer the user query.\n {format_instruction}\n{query}",
    input_variables=['query'],
    partial_variables= {"format_instruction": format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":car_query })
print(response)


{'query': 'tell me the information about Lambogini', 'response': "Lamborghini (officially Automobili Lamborghini S.p.A.) is an Italian brand and manufacturer of luxury sports cars and SUVs based in Sant'Agata Bolognese. The company is owned by the Volkswagen Group through its subsidiary Audi. Founded by Ferruccio Lamborghini in 1963, the company was created with the aim of competing with established marques like Ferrari. Lamborghini is renowned for its high-performance vehicles, often characterized by their distinctive, aggressive styling and powerful engines, typically V10 or V12. Iconic models include the Miura, Countach, Diablo, Murciélago, Gallardo, Aventador, Huracán, and the Urus SUV."}


In [19]:
user_query = "Give me details about DPO fine-tune"

output_parser = CommaSeparatedListOutputParser()
format_instruction = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query\n{format_instruction}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instruction":format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":user_query})
print(response)

['Direct Preference Optimization', 'Aligns LLMs with human preferences', 'Does not require a separate reward model', 'Uses preference data (chosen vs. rejected responses)', 'Simpler and more stable than PPO-based RLHF', 'Optimizes a direct analytical loss function', 'Based on a supervised fine-tuned (SFT) model', 'Computationally efficient', 'Achieves comparable or superior performance to RLHF', 'Avoids complexities of reinforcement learning']


## **LangChain Components for RAG Application**

### 7. Documents

#### Document Object

The document object contains two main components:

1. page-content: this represents the content of the whole document.
2. metadata: this represents the all the attributes that paticularlly unique for the each document. this availables at Dict types

examples:
document_id, timestamp, document_title, author, etc

In [20]:
document = Document(
    page_content="This is a example for document attribute which is page_content",
    metadata={
        "document_id": 1294857,
        "document_source": "about firewall",
        "created_time": 21344656
    }
)

document.page_content

'This is a example for document attribute which is page_content'

#### Document Loaders

Document loaders in LangChain are used to import data from multiple sources. For example, you can load a PDF file and make it readable for an LLM using LangChain.

LangChain provides more than 100 different document loaders, along with integrations with major platforms like AirByte and Unstructured. These integrations allow you to load various types of content such as HTML, PDFs, and source code from different locations, including private S3 buckets and public websites.

A full list of supported document types can be found here: https://python.langchain.com/v0.1/docs/integrations/document_loaders/

In [21]:
file_path = "https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf"
loader = PyPDFLoader(file_path)

In [22]:
loader

In [23]:
document = loader.load()
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tool

#### URL and Website Loaders

In [24]:
web_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()

In [25]:
web_data[0].page_content[:1000]

'LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.Copy pageDocumentation IndexFetch the comp

#### Text-Splitter

After loading documents, you often need to transform them so they work better for your specific use case.

A common step is breaking a long document into smaller chunks that can fit within a model’s context window. LangChain provides several built-in document transformers that help with splitting, merging, filtering, and other ways of processing documents.

In general, text splitters work in the following way:

1. The text is first divided into small, meaningful units, such as sentences.
2. These small parts are then gradually combined into larger chunks until a defined size limit is reached (based on a chosen measurement method).
3. Once a chunk reaches the limit, it is stored as a separate segment, and a new chunk begins. Often, a small overlap is kept between chunks to preserve context across sections.

You can view the different types of text splitters supported by LangChain here:
[https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/)

As an example, we will use the simple `CharacterTextSplitter` to divide the LangChain paper you previously loaded. This basic splitter works by breaking text using characters (defaulting to `"\n\n"`) and measuring chunk size based on the number of characters.


In [26]:
text_splitter = CharacterTextSplitter(chunk_size = 700, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))

60


In [27]:
print(chunks[8].page_content)

deployment of LangChain applications. Finally, section 5 addresses the limita-
tions and criticisms of LangChain, particularly focusing on the complexities and
security concerns associated with its modular design and external integrations.
1 Architecture
LangChain is built with a modular architecture, designed to simplify the life-
cycle of applications powered by large language models (LLMs), from initial
development through to deployment and monitoring [3]. This modularity al-
lows developers to configure, extend, and deploy applications tailored to specific


#### Embedding Models

Embedding models are designed to work directly with text embeddings.

They convert a piece of text into a numerical vector representation. This is useful because it lets you represent text in a vector space, where similar meanings are positioned closer together. As a result, you can perform tasks like semantic search by finding text segments that are closest to each other in that vector space.


In [36]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    api_key=Model_API
)

In [29]:
text = "Hi im seran"
embeds = embeddings.embed_query("Hi im seran")
embeds

[0.0056103403,
 0.036686193,
 0.007968124,
 -0.012751826,
 0.020288246,
 -0.011948124,
 0.009202696,
 0.013190522,
 0.0022924612,
 -0.036529027,
 -0.002601864,
 -0.004427653,
 -0.0040804255,
 0.00012383792,
 0.00059155736,
 -0.024476398,
 0.03420043,
 -0.0002107415,
 -0.006732871,
 -0.01092944,
 0.009508564,
 -0.02845144,
 0.007161191,
 -0.0060097887,
 -0.0053270287,
 -0.012943567,
 -0.011133201,
 -0.0049616075,
 -0.03637322,
 0.12255258,
 -0.007223886,
 0.009102838,
 0.0021087567,
 -0.02021646,
 -0.0018475761,
 -0.016638352,
 0.0013085961,
 0.0022267252,
 -0.016855603,
 -0.0024761264,
 0.0059231357,
 0.0027393845,
 0.013814153,
 0.012377565,
 -0.008859692,
 0.0071116243,
 -0.014386142,
 0.0025682084,
 -0.0029223713,
 0.007140271,
 -0.0028166755,
 -0.017668193,
 0.015596635,
 -0.014943381,
 -0.0023113927,
 -0.034170672,
 0.0021437968,
 0.002174044,
 -0.030944992,
 0.0023542081,
 -0.016099293,
 0.02163537,
 0.00972708,
 -0.0062310835,
 -0.019994,
 -0.010132473,
 -0.00037567478,
 0.03415

In [30]:
## to embedding documents
text = [doc.page_content for doc in chunks]
document_embeds = embeddings.embed_documents(text[0:100])

In [31]:
print(len(document_embeds))

60


#### Vector Stores

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) takes care of storing embedded data and performing vector search for you.


We use **Chroma** here to demonstrase the vector stores

In [ ]:
# 1st initiate chroma object using document chunks and embeddings
doc_search = Chroma.from_documents(chunks, embeddings)

query = "Langchain"

#if we want to find the relevant context for query
docs = doc_search.similarity_search(query)
docs

[Document(id='01e05657-af18-48b2-8103-a17b060c449a', metadata={'subject': '', 'moddate': '2024-11-06T10:08:55+00:00', 'trapped': '/False', 'page_label': '1', 'creationdate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'total_pages': 14, 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'author': '', 'keywords': '', 'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'page': 0, 'title': ''}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and conte

In [38]:
print(docs[0].page_content)

LangChain
Vasilios Mavroudis
Alan Turing Institute
vmavroudis@turing.ac.uk
Abstract. LangChain is a rapidly emerging framework that offers a ver-
satile and modular approach to developing applications powered by large
language models (LLMs). By leveraging LangChain, developers can sim-
plify complex stages of the application lifecycle—such as development,
productionization, and deployment—making it easier to build scalable,
stateful, and contextually aware applications. It provides tools for han-
dling chat models, integrating retrieval-augmented generation (RAG),
and offering secure API interactions. With LangChain, rapid deployment


#### Retrievers

A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. Retrievers can be created from vector stores, but are also broad enough to include other sources.
Retrievers accept a string query as input and return a list of Document objects as output.

Note that all vector stores can be cast to retrievers. Refer to the vector store integration docs for available vector stores. This page lists custom retrievers, implemented via subclassing BaseRetriever.

Define **vectore store backbone retriever and parent document retriever**

1. Vectore Store Backbone Retriever

* Note that the results are identical to the ones obtained using the similarity search strategy.


In [40]:
retriever = doc_search.as_retriever()
docs = retriever.invoke("Langchain")
docs[0]

Document(id='01e05657-af18-48b2-8103-a17b060c449a', metadata={'trapped': '/False', 'page_label': '1', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'total_pages': 14, 'author': '', 'producer': 'pdfTeX-1.40.26', 'title': '', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'page': 0, 'creationdate': '2024-11-06T10:08:55+00:00', 'subject': '', 'moddate': '2024-11-06T10:08:55+00:00', 'creator': 'LaTeX with hyperref', 'keywords': ''}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contex


#### What is Parent Document Retriever?

The Parent Document Retriever is a retrieval technique used in Retrieval-Augmented Generation (RAG) systems to balance two important requirements:

1. **Accurate retrieval** using small text chunks.
2. **Rich context** using larger documents.

It stores small chunks for vector search while returning larger parent documents during retrieval.

---

#### The Problem

When documents are split for vector databases, there is a trade-off.

#### Small Chunks

**Advantages**
- More precise embeddings
- Better semantic search accuracy

**Disadvantages**
- Limited context
- Important surrounding information may be lost

#### Large Chunks

**Advantages**
- More context for the language model

**Disadvantages**
- Less accurate embeddings
- Retrieval quality may decrease

---

#### How Parent Document Retriever Solves This

Instead of choosing one approach, it combines both.

#### Parent Document
A larger section of text containing complete context.

#### Child Chunks
Smaller pieces created from the parent document.

The vector database stores embeddings for the child chunks, while the original parent document is stored separately.

---

#### Retrieval Process

#### Step 1: Create Parent Documents

A large document is divided into logical sections.

#### Step 2: Split into Child Chunks

Each parent document is broken into smaller chunks.

#### Step 3: Store Embeddings

Embeddings are generated only for the child chunks and stored in the vector database.

#### Step 4: Perform Similarity Search

When a user submits a query, the system searches the vector database and finds the most relevant child chunk.

#### Step 5: Retrieve Parent Document

Instead of returning only the matching child chunk, the system retrieves the corresponding parent document and sends it to the LLM.

---

#### Benefits

#### Accurate Retrieval
Small chunks produce focused embeddings and improve search quality.

#### Better Context
The LLM receives a larger document section instead of an isolated sentence.

#### Improved Answer Quality
More surrounding information helps the model generate complete and accurate answers.

#### Reduced Context Loss
Important details before and after the matching chunk are preserved.

---

#### Typical Configuration

```text
Parent Chunk Size : 1000–2000 characters
Child Chunk Size  : 200–400 characters